# AI-Based Household Electricity Management Application
## Machine Learning Model Training, Comparison & Evaluation

**Project**: MCA Major Project  
**Module**: Core Research Module – Consumption Prediction Regression  
**Target Variable**: `monthly_consumption_kwh` (End-of-month electricity usage in kWh)

---
### Algorithms Implemented:
1. **Linear Regression** (Baseline parametric model)
2. **Decision Tree Regression** (Non-linear decision tree)
3. **Random Forest Regression** (Ensemble bagging)
4. **XGBoost Regression** (Gradient boosted decision trees)
5. **K-Nearest Neighbors Regression** (Instance-based distance regression)


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.abspath('..'))

from src.data.load_data import load_datasets
from src.features.feature_pipeline import (
    prepare_features,
    build_preprocessing_pipeline,
    TARGET_COLUMN,
    FEATURE_COLUMNS
)
from src.training.train_models import train_all_models
from src.evaluation.evaluate_models import evaluate_all_models, identify_best_model
from src.visualization.plot_results import generate_all_evaluation_plots
from src.inference.predict import ElectricityPredictor


### 1. Load Training and Testing Datasets

In [ ]:
base_dir = os.path.abspath('..')
train_df, test_df, files = load_datasets(base_dir)

print(f'Training observations: {len(train_df):,}, Columns: {len(train_df.columns)}')
print(f'Testing observations:  {len(test_df):,}, Columns: {len(test_df.columns)}')
train_df[['monthly_consumption_kwh', 'household_size', 'mean_temperature_c', 'season']].head()


### 2. Feature Extraction & Preprocessor Fitting (No Leakage)

In [ ]:
X_train, y_train = prepare_features(train_df, is_training=True)
X_test, y_test = prepare_features(test_df, is_training=False)

preprocessor = build_preprocessing_pipeline()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f'Processed Feature Matrix Shape: Train={X_train_proc.shape}, Test={X_test_proc.shape}')


### 3. Train Five Regression Algorithms

In [ ]:
trained_models = train_all_models(
    X_train_processed=X_train_proc,
    y_train=y_train,
    output_dir=None,
    random_state=42
)


### 4. Comparative Evaluation on Held-out Testing Dataset

In [ ]:
comparison_df, eval_records, predictions_dict = evaluate_all_models(
    trained_models=trained_models,
    X_train_processed=X_train_proc,
    y_train=y_train,
    X_test_processed=X_test_proc,
    y_test=y_test
)

comparison_df


### 5. Best Model Identification

In [ ]:
best_name, best_summary = identify_best_model(comparison_df)
print(f'Selected Best Model: {best_name}')
print(f"Test R^2:  {best_summary['Test_R2']:.4f}")
print(f"Test RMSE: {best_summary['Test_RMSE']:.4f} kWh")
print(f"Test MAE:  {best_summary['Test_MAE']:.4f} kWh")


### 6. Verify Production Inference Using Saved Artifact

In [ ]:
predictor = ElectricityPredictor(models_dir=os.path.join(base_dir, 'models'))

sample_household = test_df.iloc[0].to_dict()
result = predictor.predict(sample_household)

print('Sample Prediction Result:')
print(result)
